In [0]:
dbutils.widgets.dropdown(
    "taxi_type",
    "yellow",
    ["yellow", "green"],
    "Taxi type"
)

taxi_type = dbutils.widgets.get("taxi_type").strip().lower()

if taxi_type not in {"yellow", "green"}:
    raise ValueError("taxi_type must be yellow or green.")

print(f"Taxi type: {taxi_type}")

In [0]:
from datetime import datetime, timezone
from pathlib import Path
import requests

MAX_LOOKBACK_MONTHS = 12

now_utc = datetime.now(timezone.utc)

# Începem cu ultima lună calendaristică încheiată.
if now_utc.month == 1:
    candidate_year = now_utc.year - 1
    candidate_month = 12
else:
    candidate_year = now_utc.year
    candidate_month = now_utc.month - 1


def previous_month(year, month):
    if month == 1:
        return year - 1, 12

    return year, month - 1


selected_year = None
selected_month = None
selected_url = None
should_process = False

bronze_base_path = (
    "/Volumes/workspace/urban_mobility_bronze/landing"
)

for _ in range(MAX_LOOKBACK_MONTHS):
    period = f"{candidate_year}-{candidate_month:02d}"

    file_name = (
        f"{taxi_type}_tripdata_{period}.parquet"
    )

    source_url = (
        "https://d37ci6vzurychx.cloudfront.net/"
        f"trip-data/{file_name}"
    )

    destination_path = (
        f"{bronze_base_path}/{taxi_type}/"
        f"year={candidate_year}/"
        f"month={candidate_month:02d}/"
        f"{file_name}"
    )

    try:
        response = requests.head(
            source_url,
            allow_redirects=True,
            timeout=30,
        )

        source_exists = response.status_code == 200

    except requests.RequestException as error:
        print(
            f"Source check failed for {period}: {error}"
        )
        source_exists = False

    destination_exists = Path(destination_path).is_file()

    print(
        f"{period}: "
        f"source_exists={source_exists}, "
        f"bronze_exists={destination_exists}"
    )

    if source_exists and not destination_exists:
        selected_year = candidate_year
        selected_month = candidate_month
        selected_url = source_url
        should_process = True
        break

    candidate_year, candidate_month = previous_month(
        candidate_year,
        candidate_month,
    )

In [0]:
if should_process:
    print(
        "New source selected: "
        f"{taxi_type}/{selected_year}/{selected_month:02d}"
    )
    print(f"Source URL: {selected_url}")

else:
    # Valorile trebuie definite chiar dacă nu există date noi,
    # deoarece workflow-ul le referențiază în parametri.
    selected_year = candidate_year
    selected_month = candidate_month

    print(
        "No new published and unprocessed source "
        f"found in the last {MAX_LOOKBACK_MONTHS} months."
    )

dbutils.jobs.taskValues.set(
    key="year",
    value=selected_year,
)

dbutils.jobs.taskValues.set(
    key="month",
    value=selected_month,
)

dbutils.jobs.taskValues.set(
    key="should_process",
    value=should_process,
)

print(f"Selected year: {selected_year}")
print(f"Selected month: {selected_month}")
print(f"Should process: {should_process}")